In [54]:
from sklearn.linear_model import LassoCV
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np
import seaborn as sns

In [55]:
live = pd.read_csv("../data/samples/trial/live_metrics.csv")
verbose = pd.read_csv("../data/samples/trial/verbose_statements.csv")
initial = pd.read_csv("../data/samples/trial/initial_statement.csv")

In [56]:
live = live.drop(['Unnamed: 0', "_id", "Category", "Collect", "Current Time"], axis=1)

In [57]:
response_time = verbose['total_duration'].tolist()
eval_rate = verbose['eval_rate'].tolist()
response_time = response_time[:-1]
reset_iters = live[live['Iteration'] == 1].index.tolist()

# gpu_util_avg = []
# memory_util_avg = []
# clock_util_avg = []

# for i in range(1, len(reset_iters)):
#     temp_metrics = live.dropna()
#     gpu_util_prompt= temp_metrics.iloc[reset_iters[i-1]:reset_iters[i], 3].tolist()
#     memeory_util_prompt= temp_metrics.iloc[reset_iters[i-1]:reset_iters[i], 9].tolist()
#     clock_util_prompt= temp_metrics.iloc[reset_iters[i-1]:reset_iters[i], -2].tolist()
#     gpu_util_avg.append(np.average(gpu_util_prompt))
#     memory_util_avg.append(np.average(memeory_util_prompt))
#     clock_util_avg.append(np.average(clock_util_prompt))

In [58]:
eval_rates = [np.float64(evals[:-9]) for evals in eval_rate]

In [ ]:
live = live.dropna()
live.corr()

In [60]:
live = live.drop(columns=['Memory Clock Utilization', 'Memory Current Clock (MHz)'])

In [ ]:
mask = np.triu(np.ones_like(live.corr(), dtype=bool))
heatmap = sns.heatmap(live.corr(), mask=mask, annot=True, cmap="crest")

In [ ]:
sns.set_theme()
g = sns.PairGrid(live.iloc[:, 4:])
plot = g.map(sns.scatterplot)
# sns.pairplot(live.iloc[:, 4:], corner=True)

In [63]:
# X = pd.DataFrame([gpu_util_avg, memory_util_avg, clock_util_avg])
# X = X.T
# X.columns = ['GPU Util', "Memory Util", "GPU Clock"]
X_sam = live
# y = pd.DataFrame(response_time, columns=['AVG Response Time'])
y = np.array(response_time)

In [64]:
column_names = live.columns.tolist()
live_avg = dict(zip(column_names, [list(range(70))] * len(column_names)))
live_avg = pd.DataFrame(live_avg, dtype=np.float64)

In [65]:
for iter in range(1, len(reset_iters)):
    for column in live.columns:
        temp_list = live.iloc[reset_iters[iter-1]:reset_iters[iter],  column_names.index(column)].tolist()
        live_avg.iloc[iter-1, column_names.index(column)] = np.float64(np.average(temp_list))

In [66]:
live_avg = live_avg.iloc[:-1,:]

In [ ]:
from sklearn.preprocessing import normalize, StandardScaler

live_avg = normalize(live_avg)
ss = StandardScaler()
live_avg = ss.fit_transform(live_avg)

In [69]:
X_train, X_test, y_train, y_test = train_test_split(live_avg, y, test_size=0.2, shuffle=True, random_state=20)

In [76]:
lcv = LassoCV(cv=5)
lcv.fit(X_train, y_train)
score = lcv.score(X_test, y_test)
print(f"Accuracy: {score}")

Accuracy: -91.60183919233819


c:\Users\rahul\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:683: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.1774100330891315, tolerance: 0.16134245416918935
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\rahul\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:683: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.3203319217547005, tolerance: 0.16134245416918935
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\rahul\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:683: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.32861625220141377, tolerance: 0.16134245416918935
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\rahul\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.

In [ ]:
lcv.alpha_

In [71]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import Lasso, Ridge

rfe = RFE(estimator=Ridge(), n_features_to_select=2, verbose=True)


In [72]:
rfe.fit(X_train, y_train)
rfe.score(X_test, y_test)

Fitting estimator with 9 features.
Fitting estimator with 8 features.
Fitting estimator with 7 features.
Fitting estimator with 6 features.
Fitting estimator with 5 features.
Fitting estimator with 4 features.
Fitting estimator with 3 features.


-89.0946337089728

In [73]:
dict(zip(live.columns.tolist(), rfe.support_))

{'GPU Utilization (%)': False,
 'Power Draw (Watts)': False,
 'GPU Temp (°C)': True,
 'GPU Current Clock (MHz)': False,
 'Memory Allocation Used (MB)': False,
 'Memory Utilization (%)': False,
 'Time Delta': False,
 'Iteration': True,
 'GPU Clock Utilization': False}